In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
df = pd.read_csv('../data/raw/Works Sanctioned.csv')

In [6]:
df.head()

,Sr. No.,Work category,Work,State,IDA,Hon'ble Members of Parliament,Constituency,Work description,Recommended date,Sanction Date,Sanction Amount ( ₹ ),Work Status
0,1,Normal/Others,WS/\t MP620/2024-2025/133166-Construction of b...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan at Navalgund ...,08-Jul-2024,09-Jul-2024,497185,Physical Inspection
1,2,Trust and Society,WS/\t MP620/2025-2026/133167-Construction of r...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of College room of CBS Charitabl...,08-Jul-2024,18-Sep-2025,500000,Sanction
2,3,Trust and Society,WS/\t MP620/2024-2025/133190-Construction of b...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan of Veerashaiv...,08-Jul-2024,23-Sep-2024,450000,Sanction
3,4,Normal/Others,WS/\t MP620/2025-2026/133191-Construction of b...,Karnataka,HAVERI(DEPUTY COMMISSIONER HAVERI_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Maremma Cultural bhavan at Nee...,08-Jul-2024,29-May-2025,1500000,Work partially Completed
4,5,Normal/Others,WS/\t MP620/2024-2025/133301-Construction of b...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Cultural Bhavan at Kundagol TQ...,09-Jul-2024,22-Aug-2024,398009,Physical Inspection


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36001 entries, 0 to 36000
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Sr. No.                        36001 non-null  str  
 1   Work category                  36001 non-null  str  
 2   Work                           36001 non-null  str  
 3   State                          36001 non-null  str  
 4   IDA                            36001 non-null  str  
 5   Hon'ble Members of Parliament  36001 non-null  str  
 6   Constituency                   36001 non-null  str  
 7   Work description               35915 non-null  str  
 8   Recommended date               36001 non-null  str  
 9   Sanction Date                  36001 non-null  str  
 10  Sanction Amount ( ₹ )          36001 non-null  str  
 11  Work Status                    36001 non-null  str  
dtypes: str(12)
memory usage: 3.3 MB


In [8]:
df.isnull().sum()

Sr. No.                           0
Work category                     0
Work                              0
State                             0
IDA                               0
Hon'ble Members of Parliament     0
Constituency                      0
Work description                 86
Recommended date                  0
Sanction Date                     0
Sanction Amount ( ₹ )             0
Work Status                       0
dtype: int64

In [9]:
df.duplicated().sum()

0

In [10]:
df.columns = (
    pd.Index(df.columns)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [11]:
df.columns

Index(['sr_no', 'work_category', 'work', 'state', 'ida',
       'hon_ble_members_of_parliament', 'constituency', 'work_description',
       'recommended_date', 'sanction_date', 'sanction_amount', 'work_status'],
      dtype='str')

In [12]:
for col in df.columns:
    if col == "work":
        continue

    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.replace(r"[\t\r\n]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


In [13]:
grand_total_mask = pd.Series(False, index=df.index)

for col in df.columns:
    if col == "work":
        continue

    grand_total_mask |= df[col].str.fullmatch(
        r"grand\s+total",
        case=False,
        na=False
    )

print("Grand Total rows found:", int(grand_total_mask.sum()))

df = df.loc[~grand_total_mask].copy()

Grand Total rows found: 1


In [14]:
df = df.replace(r"^\s*$", pd.NA, regex=True)

print("Rows remaining:", len(df))

Rows remaining: 36000


In [15]:
df["work"] = df["work"].str.replace(
    r"\s*/\s*",
    "/",
    regex=True
)

In [16]:
df["work_id"] = df["work"].str.extract(
    r"(?i)(WS/MP\d+/\d{4}-\d{4}/\d+)",
    expand=False
)
df["work_id"] = df["work_id"].str.upper()

In [17]:
df["work_detail_from_reference"] = df["work"].str.extract(
    r"(?i)^WS/MP\d+/\d{4}-\d{4}/\d+-(.*)$",
    expand=False
).str.strip()

In [18]:
df.head()

,sr_no,work_category,work,state,ida,hon_ble_members_of_parliament,constituency,work_description,recommended_date,sanction_date,sanction_amount,work_status,work_id,work_detail_from_reference
0,1,Normal/Others,WS/MP620/2024-2025/133166-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan at Navalgund ...,08-Jul-2024,09-Jul-2024,497185,Physical Inspection,WS/MP620/2024-2025/133166,Construction of buildings for community cultur...
1,2,Trust and Society,WS/MP620/2025-2026/133167-Construction of room...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of College room of CBS Charitable...,08-Jul-2024,18-Sep-2025,500000,Sanction,WS/MP620/2025-2026/133167,Construction of rooms and halls in school and ...
2,3,Trust and Society,WS/MP620/2024-2025/133190-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan of Veerashaiv...,08-Jul-2024,23-Sep-2024,450000,Sanction,WS/MP620/2024-2025/133190,Construction of buildings for community cultur...
3,4,Normal/Others,WS/MP620/2025-2026/133191-Construction of buil...,Karnataka,HAVERI(DEPUTY COMMISSIONER HAVERI_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Maremma Cultural bhavan at Nee...,08-Jul-2024,29-May-2025,1500000,Work partially Completed,WS/MP620/2025-2026/133191,Construction of buildings for community cultur...
4,5,Normal/Others,WS/MP620/2024-2025/133301-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Cultural Bhavan at Kundagol TQ...,09-Jul-2024,22-Aug-2024,398009,Physical Inspection,WS/MP620/2024-2025/133301,Construction of buildings for community cultur...


In [19]:
amount_text = (
    df["sanction_amount"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("₹", "", regex=False)
    .str.strip()
)


In [20]:
missing_amount = amount_text.str.upper().isin(
    ["", "NA", "N/A", "NULL", "-"]
)
amount_text = amount_text.mask(missing_amount, pd.NA)

In [21]:
df["sanction_amount"] = pd.to_numeric(
    amount_text,
    errors="coerce"
)

In [22]:
def parse_date_column(series):
    cleaned = series.astype("string").str.strip()

    missing = cleaned.str.upper().isin(
        ["", "NA", "N/A", "NULL", "-"]
    )
    cleaned = cleaned.mask(missing, pd.NA)

    return pd.to_datetime(
        cleaned,
        format="%d-%b-%Y",
        errors="coerce"
    )


In [23]:
for col in ["recommended_date", "sanction_date"]:

    raw_dates = df[col].astype("string").str.strip()
    missing = raw_dates.str.upper().isin(
        ["", "NA", "N/A", "NULL", "-"]
    )

    parsed_dates = parse_date_column(raw_dates)

    df[col] = parsed_dates

In [24]:
df["days_recommendation_to_sanction"] = (
    df["sanction_date"] - df["recommended_date"]
).dt.days

In [25]:
(df['days_recommendation_to_sanction'] < 0).sum()

0

In [26]:
df.head()

,sr_no,work_category,work,state,ida,hon_ble_members_of_parliament,constituency,work_description,recommended_date,sanction_date,sanction_amount,work_status,work_id,work_detail_from_reference,days_recommendation_to_sanction
0,1,Normal/Others,WS/MP620/2024-2025/133166-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan at Navalgund ...,2024-07-08,2024-07-09,497185.0,Physical Inspection,WS/MP620/2024-2025/133166,Construction of buildings for community cultur...,1
1,2,Trust and Society,WS/MP620/2025-2026/133167-Construction of room...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of College room of CBS Charitable...,2024-07-08,2025-09-18,500000.0,Sanction,WS/MP620/2025-2026/133167,Construction of rooms and halls in school and ...,437
2,3,Trust and Society,WS/MP620/2024-2025/133190-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan of Veerashaiv...,2024-07-08,2024-09-23,450000.0,Sanction,WS/MP620/2024-2025/133190,Construction of buildings for community cultur...,77
3,4,Normal/Others,WS/MP620/2025-2026/133191-Construction of buil...,Karnataka,HAVERI(DEPUTY COMMISSIONER HAVERI_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Maremma Cultural bhavan at Nee...,2024-07-08,2025-05-29,1500000.0,Work partially Completed,WS/MP620/2025-2026/133191,Construction of buildings for community cultur...,325
4,5,Normal/Others,WS/MP620/2024-2025/133301-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Cultural Bhavan at Kundagol TQ...,2024-07-09,2024-08-22,398009.0,Physical Inspection,WS/MP620/2024-2025/133301,Construction of buildings for community cultur...,44


In [27]:
df.drop(columns=["work"], inplace=True)

In [ ]:
df.to_csv('../processed/works_sanctioned_cleaned.csv', index=False)